# Heart Disease Prediction — Exploratory Data Analysis

**Dataset:** UCI Heart Disease (Cleveland)  
**Author:** Ishita Singh, ECE, JIIT Noida  

---

This notebook covers:
1. Dataset loading & overview
2. Univariate & bivariate analysis
3. Correlation analysis
4. Class balance check
5. Full pipeline run with results

In [ ]:
import sys
sys.path.append('..')   # allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
from src.preprocessing import load_data

df = load_data('../data/heart_disease_uci.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(3)

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values,
            color=['#2563EB', '#DC2626'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_ylabel('Patient Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie
axes[1].pie(counts.values, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['#2563EB', '#DC2626'],
            startangle=90, pctdistance=0.8)
axes[1].set_title('Class Balance', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Feature Distributions by Target

In [ ]:
continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(continuous):
    for label, color in [(0, '#2563EB'), (1, '#DC2626')]:
        axes[i].hist(
            df[df['target'] == label][col].dropna(),
            bins=20, alpha=0.6, color=color,
            label='No Disease' if label == 0 else 'Disease'
        )
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

axes[-1].axis('off')
plt.suptitle('Feature Distributions by Target Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 5. Run Full ML Pipeline

In [ ]:
from src.preprocessing import run_preprocessing_pipeline
from src.model import train_all_models
from src.utils import evaluate_model, compare_models

X_train, X_test, y_train, y_test, scaler, feature_names = \
    run_preprocessing_pipeline('../data/heart_disease_uci.csv')

trained_models = train_all_models(X_train, y_train)

In [ ]:
results = []
for name, model in trained_models.items():
    metrics = evaluate_model(model, X_test, y_test, model_name=name)
    results.append(metrics)

comparison_df = compare_models(results)
comparison_df

In [ ]:
from src.utils import plot_roc_curve, plot_model_comparison, plot_feature_importance

plot_roc_curve(trained_models, X_test, y_test, output_dir='../outputs')
plot_model_comparison(comparison_df, output_dir='../outputs')

best = comparison_df.iloc[0]['model']
plot_feature_importance(trained_models[best], feature_names,
                        model_name=best, output_dir='../outputs')

print('\nAll plots saved to outputs/')